In [64]:
#%pip install pyobis

In [7]:
import os
import pandas as pd
from tqdm import tqdm
import pyarrow as pa
import pyarrow.parquet as pq
import glob
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
from upsetplot import UpSet, from_memberships
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import re
import unicodedata
import requests
import geopandas as gpd
from shapely.geometry import Point
import xml.etree.ElementTree as ET
import dask.dataframe as dd
import csv
import chardet
import rasterio 
import geopandas as gpd
import numpy as np
from rasterio.transform import from_bounds
from osgeo import gdal
from fastparquet import ParquetFile
from datetime import datetime
import time
import pyobis

os.chdir ('C:\\Users\\Admin\\Documents\\OBIS')

## Check column types

In [3]:
# Load the Parquet file
file_path = "merged_filtered.parquet"
df = pd.read_parquet(file_path)
# Display basic information about the dataset
print(f"Number of rows: {len(df)}")
print(f"Number of columns: {len(df.columns)}")
# Display column names and data types
print("Column names and data types:")
print(df.dtypes)
# Save column names and data types to a CSV file
output_csv = "column_data_types.csv"  # Specify the output CSV file name
column_info = pd.DataFrame({
    "Column Name": df.dtypes.index,  # Column names
    "Data Type": df.dtypes.values    # Corresponding data types
})
# Save the column information to a CSV file
column_info.to_csv(output_csv, index=False)
print(f"Column names and data types saved to {output_csv}")
# Calculate coverage (percentage of non-missing values) for each column
coverage = (df.notnull().sum() / len(df)) * 100  # Calculate percentage of non-missing values
coverage_df = pd.DataFrame({
    "Column Name": coverage.index,  # Column names
    "Coverage (%)": coverage.values  # Corresponding coverage percentages
})
# Display coverage information
print("Coverage (percentage of non-missing values) for each column:")
print(coverage_df)
# Save coverage information to a CSV file
coverage_csv = "column_coverage.csv"  # Specify the output CSV file name
coverage_df.to_csv(coverage_csv, index=False)
print(f"Coverage information saved to {coverage_csv}")

Number of rows: 619939
Number of columns: 226
Column names and data types:
basisOfRecord             object
brackish                  object
catalogNumber             object
class                     object
classid                  float64
                          ...   
roundedLongitude          object
unique_id                 object
associatedOccurrences     object
identifiedByID            object
vitality                  object
Length: 226, dtype: object
Column names and data types saved to column_data_types.csv
Coverage (percentage of non-missing values) for each column:
               Column Name  Coverage (%)
0            basisOfRecord    100.000000
1                 brackish     78.109943
2            catalogNumber     39.780204
3                    class     92.313276
4                  classid     92.313276
..                     ...           ...
221       roundedLongitude    100.000000
222              unique_id    100.000000
223  associatedOccurrences      0.053070
224  

# Data cleaning

## Filter out stations on land based on bathymetry

In [6]:
# Step 1: Load the Parquet file
input_file = "merged_filtered.parquet"
df = pd.read_parquet(input_file)
# Step 2: Extract unique station locations along with the year column
unique_stations = df[['decimalLatitude', 'decimalLongitude']].drop_duplicates()
# Step 3: Save the result to a new file
output_file = "unique_coordinates.parquet"
unique_stations.to_parquet(output_file, index=False)
# Optional: Save as CSV if needed
unique_stations.to_csv("unique_coordinates.csv", index=False)
print(f"Unique station locations with year saved to {output_file}")

Unique station locations with year saved to unique_coordinates.parquet


In [7]:
def extract_raster_values(raster_path, coordinates_df, batch_size=10000):
    """
    Extract raster values for given coordinates from a single raster file.
    Parameters:
        raster_path (str): Path to the merged raster file.
        coordinates_df (pd.DataFrame): DataFrame with 'decimalLatitude' and 'decimalLongitude'.
        batch_size (int): Number of points to process in each batch.
    Returns:
        pd.DataFrame: DataFrame with extracted raster values.
    """
    results = []
    total_batches = len(coordinates_df) // batch_size + (1 if len(coordinates_df) % batch_size != 0 else 0)
    with rasterio.open(raster_path) as src:
        bounds = src.bounds  # Get raster bounds
        with tqdm(total=total_batches, desc="Processing Batches", unit="batch") as pbar:
            for i in range(0, len(coordinates_df), batch_size):
                batch = coordinates_df.iloc[i:i + batch_size].copy()
                batch.loc[:, 'gebco'] = np.nan  # Initialize the column for raster values
                # Adjust longitudes at the antimeridian
                batch['decimalLongitude'] = batch['decimalLongitude'].apply(
                    lambda lon: -180 if lon == 180 else lon
                )
                # Filter coordinates within the raster bounds
                batch_in_bounds = batch[
                    (batch['decimalLongitude'] >= bounds.left) &
                    (batch['decimalLongitude'] <= bounds.right) &
                    (batch['decimalLatitude'] >= bounds.bottom) &
                    (batch['decimalLatitude'] <= bounds.top)
                ]
                # Extract raster values for these coordinates
                coords = [(lon, lat) for lon, lat in zip(batch_in_bounds['decimalLongitude'], batch_in_bounds['decimalLatitude'])]
                values = list(src.sample(coords))
                # Assign values to the batch DataFrame
                for idx, value in zip(batch_in_bounds.index, values):
                    if value[0] is not None and not np.isnan(value[0]):
                        batch.loc[idx, 'gebco'] = value[0]
                results.append(batch)
                pbar.update(1)
    return pd.concat(results, ignore_index=True)
# File path to the merged raster
raster_file = r"..\Hanna\gebco_merged.tif"
# Load the unique_coordinates.parquet file
coordinates = pd.read_parquet("unique_coordinates.parquet")
# Ensure decimalLongitude and decimalLatitude are numeric
coordinates['decimalLongitude'] = pd.to_numeric(coordinates['decimalLongitude'], errors='coerce')
coordinates['decimalLatitude'] = pd.to_numeric(coordinates['decimalLatitude'], errors='coerce')
# Drop rows with invalid (NaN) coordinates
coordinates = coordinates.dropna(subset=['decimalLongitude', 'decimalLatitude'])
# Extract raster values
result_df = extract_raster_values(raster_file, coordinates)
# Save the results to a Parquet file
result_df.to_parquet("gebco_values.parquet", index=False)
# Save the results to a CSV file
result_df.to_csv("gebco_values.csv", index=False)
print("Results saved as 'gebco_values.parquet' and 'gebco_values.csv'.")

Processing Batches: 100%|██████████| 5/5 [00:08<00:00,  1.75s/batch]


Results saved as 'gebco_values.parquet' and 'gebco_values.csv'.


## Apply filter (remove records above zero)

In [9]:
# Load the Parquet file
coordinates = pd.read_parquet("gebco_values.parquet")
# Filter out stations with 'gebco' values above zero
filtered_coordinates = coordinates[coordinates['gebco'] <= 0]
# Save the filtered DataFrame to a new Parquet file
filtered_coordinates.to_parquet("gebco_filtered.parquet", index=False)
# Save the filtered DataFrame to a new CSV file
filtered_coordinates.to_csv("gebco_filtered.csv", index=False)
print("Filtered results saved as 'gebco_filtered.parquet' and 'gebco_filtered.csv'.")

Filtered results saved as 'gebco_filtered.parquet' and 'gebco_filtered.csv'.


In [10]:
# File paths
filtered_coordinates_file = "gebco_filtered.parquet"
gbif_file = "merged_filtered.parquet"
output_file = "obis_filtered_dataset_marine.parquet"
# Load the filtered coordinates (from gebco_values_filtered.parquet)
filtered_coordinates = pd.read_parquet(filtered_coordinates_file)
# Extract unique coordinate pairs and their corresponding 'gebco' values
filtered_coords_with_gebco = filtered_coordinates[['decimalLongitude', 'decimalLatitude', 'gebco']]
# Convert to a set for fast filtering
filtered_coords_set = set(zip(filtered_coords_with_gebco['decimalLongitude'], filtered_coords_with_gebco['decimalLatitude']))
# Define a function to filter a chunk and add the 'gebcoDepth' column
def filter_and_add_gebco(chunk, coords_with_gebco):
    """
    Filter a chunk of the GBIF dataset and add the 'gebcoDepth' column.
    Parameters:
        chunk (pd.DataFrame): A chunk of the GBIF dataset.
        coords_with_gebco (pd.DataFrame): DataFrame with 'decimalLongitude', 'decimalLatitude', and 'gebco'.
    Returns:
        pd.DataFrame: The filtered chunk with the 'gebcoDepth' column added.
    """
    # Convert 'decimalLongitude' and 'decimalLatitude' to numeric (float64) in the chunk
    chunk['decimalLongitude'] = pd.to_numeric(chunk['decimalLongitude'], errors='coerce')
    chunk['decimalLatitude'] = pd.to_numeric(chunk['decimalLatitude'], errors='coerce')
    # Create a tuple of (longitude, latitude) for each row in the chunk
    chunk_coords = zip(chunk['decimalLongitude'], chunk['decimalLatitude'])
    # Filter rows where the coordinate pair is in the filtered set
    filtered_chunk = chunk[[coord in filtered_coords_set for coord in chunk_coords]]
    # Merge the filtered chunk with the 'gebco' values
    filtered_chunk = filtered_chunk.merge(
        coords_with_gebco,
        on=['decimalLongitude', 'decimalLatitude'],
        how='left'
    )
    # Rename the 'gebco' column to 'gebcoDepth'
    filtered_chunk.rename(columns={'gebco': 'gebcoDepth'}, inplace=True)
    return filtered_chunk
# Open the GBIF dataset using fastparquet
pf = ParquetFile(gbif_file)
# Initialize counters
total_records_before = 0
total_records_after = 0
# Process each row group
filtered_chunks = []
for chunk in pf.iter_row_groups():
    total_records_before += len(chunk)
    
    # Filter the chunk and add the 'gebcoDepth' column
    filtered_chunk = filter_and_add_gebco(chunk, filtered_coords_with_gebco)
    filtered_chunks.append(filtered_chunk)
    total_records_after += len(filtered_chunk)
# Concatenate all filtered chunks into a single DataFrame
filtered_gbif = pd.concat(filtered_chunks, ignore_index=True)
# Save the filtered dataset to a new Parquet file
filtered_gbif.to_parquet(output_file, index=False)
# Save the filtered dataset to a new CSV file (optional)
filtered_gbif.to_csv(output_file.replace(".parquet", ".csv"), index=False)
print(f"Total records in the original OBIS dataset: {total_records_before}")
print(f"Total records after filtering: {total_records_after}")
print(f"Filtered OBIS dataset with 'gebcoDepth' saved as '{output_file}' and '{output_file.replace('.parquet', '.csv')}'.")

Total records in the original OBIS dataset: 619939
Total records after filtering: 574367
Filtered OBIS dataset with 'gebcoDepth' saved as 'obis_filtered_dataset_marine.parquet' and 'obis_filtered_dataset_marine.csv'.


# Which phyla have records on phylum level? 

In [12]:
# Load the dataset
file_path = "obis_filtered_dataset_marine.parquet"
df = pd.read_parquet(file_path)
# List of aphiaID values (as integers)
phyla = [
    882, 51, 1065, 1806, 1267, 558, 146142, 1821, 1803, 799, 
    152391, 1276, 793, 1818, 101060, 101063, 101061, 1789
]
# Filter the dataset to include only the specified acceptedTaxonKey values
filtered_df = df[df['aphiaID'].isin(phyla)]
# Filter records where 'class' is missing (null or empty)
missing_class_df = filtered_df[filtered_df['class'].isna() | (filtered_df['class'] == '')]
# Group by 'acceptedTaxonKey' and count the number of records per key
taxon_key_counts = missing_class_df.groupby('aphiaID').size().reset_index(name='record_count')
# Sort by the number of records in descending order
taxon_key_counts = taxon_key_counts.sort_values(by='record_count', ascending=False)
# Print the result
print(taxon_key_counts)

    aphiaID  record_count
6      1267         11349
10     1806          4495
16   152391          4145
15   146142          1368
3       799           932
9      1803           511
4       882           424
1       558           374
0        51           323
2       793           126
11     1818           109
14   101063            15
8      1789            14
5      1065             8
13   101060             8
7      1276             6
12     1821             2


1267: Cnidaria, 152391: Nemertea, 1806: Echinodermata, 558: Porifera, 146142: Bryozoa, 799: Nematoda, 1803: Brachiopoda, 51: Mollusca, 882: Annelida, 793: Platyhelminthes, 1818: Hemichordata, 101063: Priapulida, 1789: Phoronida, 1821: Chordata, 1065: Arthropoda, 101060: Kinorhyncha, 1276: Tardigrada

## Filter out records on phylum level for Arthropoda

In [15]:
# Load the dataset
file_path = "obis_filtered_dataset_marine.parquet"
df = pd.read_parquet(file_path)
# Filter out rows where aphiaID is 54
filtered_df = df[df['aphiaID'] != 1065]
# Save the filtered DataFrame to a new file
filtered_file_path = "obis_filtered_dataset.parquet"
filtered_df.to_parquet(filtered_file_path, index=False)
print(f"Filtered dataset saved as '{filtered_file_path}'")
# Optional: Verify the filtering
print(f"Original dataset size: {len(df)}")
print(f"Filtered dataset size: {len(filtered_df)}")

Filtered dataset saved as 'obis_filtered_dataset.parquet'
Original dataset size: 574367
Filtered dataset size: 574359


## Filter out records on phylum level for Mollusca

In [17]:
# Load the dataset
file_path = "obis_filtered_dataset.parquet"
df = pd.read_parquet(file_path)
# Filter out rows where aphiaID is 54
filtered_df = df[df['aphiaID'] != 51]
# Save the filtered DataFrame to a new file
filtered_file_path = "obis_filtered_dataset.parquet"
filtered_df.to_parquet(filtered_file_path, index=False)
print(f"Filtered dataset saved as '{filtered_file_path}'")
# Optional: Verify the filtering
print(f"Original dataset size: {len(df)}")
print(f"Filtered dataset size: {len(filtered_df)}")

Filtered dataset saved as 'obis_filtered_dataset.parquet'
Original dataset size: 574359
Filtered dataset size: 574036


## Filter out records on phylum level for Cnidaria

In [19]:
# Load the dataset
file_path = "obis_filtered_dataset.parquet"
df = pd.read_parquet(file_path)
# Filter out rows where aphiaID is 54
filtered_df = df[df['aphiaID'] != 1267]
# Save the filtered DataFrame to a new file
filtered_file_path = "obis_filtered_dataset.parquet"
filtered_df.to_parquet(filtered_file_path, index=False)
print(f"Filtered dataset saved as '{filtered_file_path}'")
# Optional: Verify the filtering
print(f"Original dataset size: {len(df)}")
print(f"Filtered dataset size: {len(filtered_df)}")

Filtered dataset saved as 'obis_filtered_dataset.parquet'
Original dataset size: 574036
Filtered dataset size: 562687


## Filter out records on phylum level for Chordata

In [21]:
# Load the dataset
file_path = "obis_filtered_dataset.parquet"
df = pd.read_parquet(file_path)
# Filter out rows where aphiaID is 54
filtered_df = df[df['aphiaID'] != 1821]
# Save the filtered DataFrame to a new file
filtered_file_path = "obis_filtered_dataset.parquet"
filtered_df.to_parquet(filtered_file_path, index=False)
print(f"Filtered dataset saved as '{filtered_file_path}'")
# Optional: Verify the filtering
print(f"Original dataset size: {len(df)}")
print(f"Filtered dataset size: {len(filtered_df)}")

Filtered dataset saved as 'obis_filtered_dataset.parquet'
Original dataset size: 562687
Filtered dataset size: 562685


## Verify

In [23]:
# Load the dataset
file_path = "obis_filtered_dataset.parquet"
df = pd.read_parquet(file_path)
# List of aphiaID values (as integers)
phyla = [
    882, 51, 1065, 1806, 1267, 558, 146142, 1821, 1803, 799, 
    152391, 1276, 793, 1818, 101060, 101063, 101061, 1789
]
# Filter the dataset to include only the specified acceptedTaxonKey values
filtered_df = df[df['aphiaID'].isin(phyla)]
# Filter records where 'class' is missing (null or empty)
missing_class_df = filtered_df[filtered_df['class'].isna() | (filtered_df['class'] == '')]
# Group by 'acceptedTaxonKey' and count the number of records per key
taxon_key_counts = missing_class_df.groupby('aphiaID').size().reset_index(name='record_count')
# Sort by the number of records in descending order
taxon_key_counts = taxon_key_counts.sort_values(by='record_count', ascending=False)
# Print the result
print(taxon_key_counts)

    aphiaID  record_count
7      1806          4495
12   152391          4145
11   146142          1368
2       799           932
6      1803           511
3       882           424
0       558           374
1       793           126
8      1818           109
10   101063            15
5      1789            14
9    101060             8
4      1276             6


# Sanity checks

## Check for duplicates

In [26]:
# Load the dataset
file_path = "obis_filtered_dataset.parquet"
df = pd.read_parquet(file_path)
# Find duplicate rows based on the 'id' column
duplicates = df[df.duplicated(subset='id', keep=False)]
# Count the number of duplicate rows
num_duplicates = len(duplicates)
# Sort the duplicates by 'id' (and optionally other columns)
duplicates_sorted = duplicates.sort_values(by='id')
# Optionally save the sorted duplicate rows to a CSV file for further inspection
duplicates_sorted.to_csv("duplicate_obis_records.csv", index=False)
# Print the number of duplicate rows and confirmation of saved file
print(f"Number of duplicate rows: {num_duplicates}")
print(f"Sorted duplicate records saved to 'duplicate_obis_records.csv'")

Number of duplicate rows: 0
Sorted duplicate records saved to 'duplicate_obis_records.csv'


## Coordinates

In [28]:
# Check latitude and longitude ranges
if "decimalLatitude" in df.columns and "decimalLongitude" in df.columns:
    print("Latitude and Longitude ranges:")
    print(f"Latitude: {df['decimalLatitude'].min()} to {df['decimalLatitude'].max()}")
    print(f"Longitude: {df['decimalLongitude'].min()} to {df['decimalLongitude'].max()}")

Latitude and Longitude ranges:
Latitude: 50.03 to 90.0
Longitude: -180.0 to 179.95425415


## Temporal coverage

In [30]:
# Check if the column exists in the DataFrame
if 'extractedEventYear' in df.columns:
    # Calculate the minimum and maximum values, ignoring NaN values
    min_year = df['extractedEventYear'].min()
    max_year = df['extractedEventYear'].max()
    
    # Print the results
    print(f"Minimum extractedEventYear: {min_year}")
    print(f"Maximum extractedEventYear: {max_year}")
else:
    print("Column 'extractedEventYear' does not exist in the DataFrame.")

Minimum extractedEventYear: 1858.0
Maximum extractedEventYear: 2025.0


## Unique values

In [5]:
# Load the Parquet file into a DataFrame
file_path = "obis_filtered_dataset.parquet"
df = pd.read_parquet(file_path)
# Calculate the number of unique entries in each column
unique_counts = df.nunique()
# Convert the results to a DataFrame
unique_counts_df = unique_counts.reset_index()
unique_counts_df.columns = ['Column', 'Unique_Count']
# Save the results to a CSV file
output_csv = "unique_counts_per_column.csv"
unique_counts_df.to_csv(output_csv, index=False)
print(f"Unique counts saved to {output_csv}")

Unique counts saved to unique_counts_per_column.csv


# Taxon counts

In [44]:
# Specify the categorical columns
categorical_columns = ["phylum", "class", "order", "family"]
# Loop through each column and save value counts as a CSV file
for col in categorical_columns:
    if col in df.columns:
        print(f"\nProcessing column: {col}")
        
        # Get value counts for the column
        value_counts = df[col].value_counts().reset_index()
        value_counts.columns = [col, "count"]  # Rename columns for clarity
        
        # Save to a CSV file
        output_file = f"record_counts_{col}.csv"
        value_counts.to_csv(output_file, index=False)
        
        print(f"Record counts for column '{col}' saved to {output_file}")


Processing column: phylum
Record counts for column 'phylum' saved to record_counts_phylum.csv

Processing column: class
Record counts for column 'class' saved to record_counts_class.csv

Processing column: order
Record counts for column 'order' saved to record_counts_order.csv

Processing column: family
Record counts for column 'family' saved to record_counts_family.csv


# Go through bibliographicCitation

In [8]:
# Load the Parquet file into a DataFrame
file_path = "obis_filtered_dataset.parquet"
df = pd.read_parquet(file_path)
# Extract unique entries from the 'bibliographicCitation' column
unique_bibliographic_citations = df['bibliographicCitation'].dropna().unique()
# Convert the unique entries to a DataFrame
unique_citations_df = pd.DataFrame(unique_bibliographic_citations, columns=['bibliographicCitation'])
# Save the results to a CSV file
output_csv = "unique_bibliographic_citations.csv"
unique_citations_df.to_csv(output_csv, index=False)
print(f"Unique bibliographic citations saved to {output_csv}")

Unique bibliographic citations saved to unique_bibliographic_citations.csv


In [3]:
# Open the file in binary mode
with open('unique_bibliographic_citations.csv', 'rb') as file:
    raw_data = file.read(10000)  # Read a portion of the file
    result = chardet.detect(raw_data)
    print(f"Encoding: {result['encoding']}, Confidence: {result['confidence']}")

Encoding: utf-8, Confidence: 0.99


In [ ]:
# Add a stopping point
print("Execution paused. Make manual adjustments and press Enter to continue.")
input()

Go through manually and save as unique_bibliographic_citations_manual.xlsx

In [22]:
# Load the Excel file
excel_df = pd.read_excel("unique_bibliographic_citations_manual.xlsx")
# Load the Parquet file
parquet_df = pd.read_parquet("obis_filtered_dataset.parquet")
# Perform the join on 'bibliographicCitation'
# Assuming 'bibliographicCitation' is the common column
merged_df = parquet_df.merge(excel_df[['bibliographicCitation', 'keep']], 
                             on='bibliographicCitation', 
                             how='left')
# Save the updated Parquet file
merged_df.to_parquet("obis_filtered_dataset.parquet", index=False)
print("The 'keep' column has been updated and saved to 'obis_filtered_dataset.parquet'.")

The 'keep' column has been updated and saved to 'obis_filtered_dataset.parquet'.


In [12]:
print(merged_df[['bibliographicCitation', 'keep']].head(10))

                               bibliographicCitation keep
0  Department of Invertebrate Zoology, Research a...    P
1                                               None  NaN
2  Continuous Plankton Recorder (CPR) data from t...    R
3  Continuous Plankton Recorder (CPR) data from t...    R
4                                               None  NaN
5                                               None  NaN
6                                               None  NaN
7                                               None  NaN
8                                               None  NaN
9                                               None  NaN


In [23]:
# Load the updated Parquet file
df = pd.read_parquet("obis_filtered_dataset.parquet")
# Subset for 'F'
subset_F = df[df['keep'] == 'F']
# Subset for 'R'
subset_R = df[df['keep'] == 'R']
# Subset for 'U'
subset_U = df[df['keep'] == 'U']
# Subset for 'P' (including NaN values)
subset_P = df[(df['keep'] == 'P') | (df['keep'].isna())]
# Save the subsets to separate Parquet files (optional)
subset_F.to_parquet("subset_F.parquet", index=False)
subset_R.to_parquet("subset_R.parquet", index=False)
subset_U.to_parquet("subset_U.parquet", index=False)
subset_P.to_parquet("subset_P.parquet", index=False)
print("Subsets created and saved successfully.")

Subsets created and saved successfully.


# Extract all unique occurrenceID and eventID

In [47]:
# Download event core and eMoF extension

In [21]:
# Load the subsets
subset_F = pd.read_parquet("subset_F.parquet")
subset_P = pd.read_parquet("subset_P.parquet")
# Combine F and P subsets
combined_subset = pd.concat([subset_F, subset_P])
# Extract unique eventID and occurrenceID
unique_event_ids = combined_subset['eventID'].dropna().unique()
unique_occurrence_ids = combined_subset['occurrenceID'].dropna().unique()
# Convert unique IDs to DataFrame for saving to CSV
unique_event_ids_df = pd.DataFrame(unique_event_ids, columns=["eventID"])
unique_occurrence_ids_df = pd.DataFrame(unique_occurrence_ids, columns=["occurrenceID"])
# Save to CSV files
unique_event_ids_df.to_csv("unique_event_ids.csv", index=False)
unique_occurrence_ids_df.to_csv("unique_occurrence_ids.csv", index=False)
print(f"Unique eventIDs: {len(unique_event_ids)}")
print(f"Unique occurrenceIDs: {len(unique_occurrence_ids)}")

Unique eventIDs: 49141
Unique occurrenceIDs: 463281
